In our last session you compared three ways to find a duplicate ID: brute force, sort, and a set.
Let us execute all 3 solutions and measure them on different sizes of input data.

In [ ]:
import random, time

def dup_brute(ids):
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            if ids[i] == ids[j]:
                return True
    return False

def dup_sort(ids):
    s = sorted(ids)
    return any(s[i] == s[i - 1] for i in range(1, len(s)))

def dup_set(ids):
    seen = set()
    for x in ids:
        if x in seen:
            return True
        seen.add(x)
    return False

In [ ]:
sizes = [500, 1000, 2000, 4000, 6000, 8000, 10000]
timings = {"brute force": [], "sort": [], "set": []}

for n in sizes:
    ids = random.sample(range(10**7), n)
    for name, f in [("brute force", dup_brute), ("sort", dup_sort), ("set", dup_set)]:
        start = time.perf_counter()
        f(ids)
        timings[name].append(time.perf_counter() - start)

print(f"{'n':>6} {'brute force':>12} {'sort':>10} {'set':>10}   (seconds)")
for k, n in enumerate(sizes):
    print(f"{n:>6} {timings['brute force'][k]:>12.5f} {timings['sort'][k]:>10.5f} {timings['set'][k]:>10.5f}")

In [ ]:
timings

Which one grows fastest?
How can we showcase or present this data?

In [ ]:
import matplotlib.pyplot as plt

for name, values in timings.items():
    plt.plot(sizes, values, marker="o", label=name)

plt.xlabel("Number of IDs")
plt.ylabel("Time (seconds)")
plt.title("Duplicate check: time as the list grows")
plt.legend()
plt.show()

The brute-force line climbs steeply, and the other two look flat. **A chart shows in a second what a table hides.**

Everything we have built so far produces data.
Now we learn to read it. And real data does not come as a Python list. It comes as **files**.


# Problem: Smoothing Seasonal Airline Traffic
- Scenario: Airline passenger numbers spike every summer and drop every winter, making it hard to see long-term growth from raw monthly numbers alone. To reveal the underlying growth trend, data analysts calculate a 12-month moving average.

- Dataset: flights.csv (144 monthly passenger counts from January 1949 to December 1960).

- Task: Write a function moving_average_12m(data, k=12) that takes a list of monthly passenger counts and returns a list of every consecutive 12-month average.

- Constraint: Your solution must run in O(N) time.

- Sample Check:
  - First 12-month average (Jan–Dec 1949): 126.67
  - Last 12-month average (Jan–Dec 1960): 476.17


## **Read a Data file**

In [ ]:
import pandas as pd

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
# read data file


In [ ]:
# view data file


In [ ]:
# How much data points we have?


In [ ]:
# What type of data fields we have?


In [ ]:
# Check first few rows of data


In [ ]:
# Check last few rows of data


In [ ]:
# Check first 10 passengers data


In [ ]:
# Check passengers and month columns


In [ ]:
# what is type of each column?


In [ ]:
passengers = df["passengers"].tolist()

In [ ]:
def moving_average_12m(data, k=12):
    window_sum = sum(data[:k])
    averages = [window_sum / k]

    for i in range(k, len(data)):
        window_sum += data[i] - data[i - k]  # Slide the window
        averages.append(window_sum / k)
    return averages

Instead of re-summing 12 months from scratch at every step O(N * K), a sliding window adds the incoming month and subtracts the outgoing month in O(N) time.

In [ ]:
ma_12 = moving_average_12m(passengers)
print(f"First 12-month average (1949): {ma_12[0]:.2f}")
print(f"Last 12-month average (1960):  {ma_12[-1]:.2f}")


# Problem: Instant Historical Flight Queries

- Scenario: You are building an internal analytics dashboard for an aviation historian. Users frequently request the total number of passengers flown between any two given months (e.g., "Total passengers from Jan 1949 to Dec 1949" or "Total passengers from Jul 1952 to Jun 1955"), resulting in thousands of repeated queries on the same dataset.

- Dataset: flights.csv (144 rows indexed 0 to 143, where 0 is Jan 1949 and 143 is Dec 1960).

- Task: Preprocess the passengers column so you can implement a function query_passengers(start_idx, end_idx) that returns the inclusive sum of passengers between start_idx and end_idx.

- Constraint: Preprocessing may take O(N) time, but each call to query_passengers must run in O(1) constant time without loops or list slicing.

- Sample Check:
	- query_passengers(0, 11) (All of 1949) → 1520
	- query_passengers(132, 143) (All of 1960) → 5714
